# Dropout implementation — 24CS2032

**Paper:** Srivastava, Nitish, et al. "Dropout: A Simple Way to Prevent Neural Networks from Overfitting." *Journal of Machine Learning Research* 15.56 (2014): 1929-1958.

**Context & Scope:** This notebook implements an educational, reduced-scale reproduction of the fixed-width dropout ablation inspired by Figure 9(a) of the original paper. Architecture: `784 → 256 → 256 → 256 → 10` MLP with ReLU activations, Adam optimizer (`lr=0.001`), 10,000 training examples, 5,000 validation examples, and 20 fixed epochs.

**Dropout Formulation:** Custom inverted `ManualDropout` after hidden ReLUs only:
- Drop probability: $d \in [0, 1)$
- Retention / keep probability: $q = 1 - d$
- Training mode: $y = x \odot \text{mask} / q$
- Evaluation mode: $y = x$
- Model contains NO built-in PyTorch dropout (`nn.Dropout` / `F.dropout`).

**Protocol & Model Selection Policy:**
- Model selection is strictly governed by lowest mean validation error across seeds at fixed epoch 20 (ties broken by lower $d$).
- Official test set (10,000 examples) is evaluated strictly for descriptive comparison; test curves are NEVER used as tuning signals.
- Self-contained notebook embedding the complete project `run.py`; no GitHub clone is required.
- No results are pre-filled as fake execution output; run cells sequentially.

In [ ]:
%pip install torch torchvision numpy pandas matplotlib


In [ ]:
from pathlib import Path
# Write exact project run.py locally
Path("run.py").write_text('import argparse\nimport gzip\nimport hashlib\nimport json\nimport math\nimport platform\nimport random\nimport shutil\nimport time\nfrom pathlib import Path\n\nimport matplotlib\nmatplotlib.use(\'Agg\')\nimport matplotlib.pyplot as plt\nimport numpy as np\nimport pandas as pd\nimport torch\nfrom torch import nn\nfrom torch.utils.data import DataLoader, TensorDataset\nfrom torchvision import datasets\nfrom torchvision.datasets.utils import check_integrity\nimport torchvision\n\n\nclass ManualDropout(nn.Module):\n    def __init__(self, drop_rate=0.5, inverted=True):\n        super().__init__()\n        if not 0 <= drop_rate < 1:\n            raise ValueError(\'drop_rate must be in [0, 1).\')\n        self.drop_rate = float(drop_rate)\n        self.inverted = inverted\n\n    def forward(self, x):\n        keep = 1.0 - self.drop_rate\n        if not self.training:\n            return x if self.inverted else x * keep\n        if self.drop_rate == 0:\n            return x\n        mask = (torch.rand_like(x) < keep).to(x.dtype)\n        return x * mask / keep if self.inverted else x * mask\n\n\nclass MLP(nn.Module):\n    def __init__(self, drop_rate, width=256):\n        super().__init__()\n        self.net = nn.Sequential(\n            nn.Flatten(),\n            nn.Linear(784, width), nn.ReLU(), ManualDropout(drop_rate),\n            nn.Linear(width, width), nn.ReLU(), ManualDropout(drop_rate),\n            nn.Linear(width, width), nn.ReLU(), ManualDropout(drop_rate),\n            nn.Linear(width, 10),\n        )\n\n    def forward(self, x):\n        return self.net(x)\n\n\ndef set_seed(seed):\n    random.seed(seed)\n    np.random.seed(seed)\n    torch.manual_seed(seed)\n    if torch.cuda.is_available():\n        torch.cuda.manual_seed_all(seed)\n        torch.backends.cudnn.benchmark = False\n        torch.backends.cudnn.deterministic = True\n\n\ndef choose_device(name):\n    if name == \'auto\':\n        name = (\'cuda\' if torch.cuda.is_available() else\n                \'mps\' if torch.backends.mps.is_available() else \'cpu\')\n    if name == \'mps\' and not torch.backends.mps.is_available():\n        raise RuntimeError(\'MPS unavailable; rerun with --device cpu.\')\n    if name == \'cuda\' and not torch.cuda.is_available():\n        raise RuntimeError(\'CUDA unavailable; use --device cpu or auto.\')\n    return torch.device(name)\n\n\ndef self_test(device):\n    set_seed(123)\n    x = torch.ones(200000, device=device, requires_grad=True)\n    layer = ManualDropout(0.3).to(device)\n    layer.train()\n    y = layer(x)\n    assert y.shape == x.shape and y.dtype == x.dtype and y.device == x.device\n    assert abs((y == 0).float().mean().item() - 0.3) < 0.01\n    assert abs(y.mean().item() - 1.0) < 0.02\n    assert torch.allclose(y[y != 0], torch.full_like(y[y != 0], 1 / 0.7))\n    y.sum().backward()\n    assert torch.allclose(x.grad, y.detach())\n    assert not torch.equal(y.detach(), layer(x).detach())\n    layer.eval()\n    assert torch.equal(layer(x), x)\n    assert torch.equal(layer(x), layer(x))\n    assert torch.equal(ManualDropout(0).to(device)(x), x)\n    classic = ManualDropout(0.3, inverted=False).to(device)\n    classic.eval()\n    assert torch.allclose(classic(x), x * 0.7)\n    classic.train()\n    assert abs(classic(x).mean().item() - 0.7) < 0.02\n    for bad in [-0.1, 1.0, 1.2]:\n        try:\n            ManualDropout(bad)\n        except ValueError:\n            pass\n        else:\n            raise AssertionError(\'Invalid rate accepted\')\n    model = MLP(0.5).to(device).eval()\n    with torch.no_grad():\n        assert model(torch.zeros(2, 1, 28, 28, device=device)).shape == (2, 10)\n    print(\'PASS: mask, scaling, gradient, fresh masks, eval, p=0, bounds, MLP shape\')\n\n\ndef load_data(name, train_n, val_n, test_n, split_seed, out):\n    cls = datasets.MNIST if name == \'mnist\' else datasets.FashionMNIST\n    def get_dataset(training):\n        try:\n            return cls(\'data\', train=training, download=True)\n        except (ValueError, RuntimeError, OSError, EOFError):\n            # Repair an interrupted empty extraction only from checksum-valid data.\n            raw = Path(\'data\') / cls.__name__ / \'raw\'\n            repaired = False\n            for filename, md5 in cls.resources:\n                archive = raw / filename\n                target = archive.with_suffix(\'\')\n                if (not target.exists() or target.stat().st_size == 0) and check_integrity(str(archive), md5):\n                    temporary = target.with_suffix(\'.repair\')\n                    with gzip.open(archive, \'rb\') as src, temporary.open(\'wb\') as dst:\n                        shutil.copyfileobj(src, dst)\n                    temporary.replace(target)\n                    repaired = True\n            if not repaired:\n                raise\n            return cls(\'data\', train=training, download=False)\n    original_train = get_dataset(True)\n    original_test = get_dataset(False)\n    if train_n + val_n > len(original_train) or test_n > len(original_test):\n        raise ValueError(\'Requested split exceeds dataset size.\')\n    gen = torch.Generator().manual_seed(split_seed)\n    order = torch.randperm(len(original_train), generator=gen)\n    # Validation and training are disjoint; official test is never used for fitting.\n    val_ids = order[:val_n]\n    train_ids = order[val_n:val_n + train_n]\n    test_ids = torch.arange(test_n)\n    np.savez(out / f\'{name}_split.npz\', train=train_ids.numpy(),\n             validation=val_ids.numpy(), test=test_ids.numpy())\n\n    def convert(ds, ids):\n        x = ds.data[ids].unsqueeze(1).float().div(255.0)\n        y = ds.targets[ids].long()\n        return TensorDataset(x, y)\n\n    sets = (convert(original_train, train_ids), convert(original_train, val_ids),\n            convert(original_test, test_ids))\n    counts = {label: torch.bincount(ds.tensors[1], minlength=10).tolist()\n              for label, ds in zip([\'train\', \'validation\', \'test\'], sets)}\n    (out / f\'{name}_class_counts.json\').write_text(json.dumps(counts, indent=2))\n    return sets, original_test.classes\n\n\n@torch.no_grad()\ndef evaluate(model, loader, device):\n    model.eval()\n    loss_sum, correct, count = 0.0, 0, 0\n    for x, y in loader:\n        x, y = x.to(device), y.to(device)\n        logits = model(x)\n        loss_sum += nn.functional.cross_entropy(logits, y, reduction=\'sum\').item()\n        correct += (logits.argmax(1) == y).sum().item()\n        count += y.numel()\n    return loss_sum / count, 100.0 * (1 - correct / count)\n\n\ndef train_one(name, rate, seed, sets, args, device, out):\n    tag = f\'{name}_d{rate:.1f}_s{seed}\'\n    checkpoint = out / f\'{tag}.pt\'\n    history_path = out / f\'{tag}.csv\'\n    if checkpoint.exists() and history_path.exists():\n        old = pd.read_csv(history_path)\n        if len(old) == args.epochs and int(old.iloc[-1][\'epoch\']) == args.epochs:\n            print(\'Already complete:\', tag)\n            return old\n    set_seed(seed)\n    model = MLP(rate, args.width).to(device)\n    optimizer = torch.optim.Adam(model.parameters(), lr=args.lr, weight_decay=0)\n    # Dedicated generator makes shuffle order identical across drop rates per seed.\n    shuffle_gen = torch.Generator().manual_seed(seed + 10000)\n    train_loader = DataLoader(sets[0], batch_size=args.batch_size, shuffle=True,\n                              generator=shuffle_gen, num_workers=0)\n    eval_loaders = [DataLoader(ds, batch_size=512, shuffle=False, num_workers=0)\n                    for ds in sets]\n    rows = []\n    start = time.perf_counter()\n    for epoch in range(1, args.epochs + 1):\n        model.train()\n        online_loss, seen = 0.0, 0\n        for x, y in train_loader:\n            x, y = x.to(device), y.to(device)\n            optimizer.zero_grad(set_to_none=True)\n            loss = nn.functional.cross_entropy(model(x), y)\n            if not torch.isfinite(loss):\n                raise RuntimeError(\'Non-finite loss. Stop and inspect configuration.\')\n            loss.backward()\n            optimizer.step()\n            online_loss += loss.item() * y.numel()\n            seen += y.numel()\n        row = dict(dataset=name, drop_rate=rate, seed=seed, epoch=epoch,\n                   updates=epoch * math.ceil(len(sets[0]) / args.batch_size),\n                   online_loss=online_loss / seen)\n        for label, loader in zip([\'train\', \'val\', \'test\'], eval_loaders):\n            row[f\'{label}_loss\'], row[f\'{label}_error\'] = evaluate(model, loader, device)\n        row[\'gap_pp\'] = row[\'test_error\'] - row[\'train_error\']\n        row[\'seconds\'] = time.perf_counter() - start\n        rows.append(row)\n        pd.DataFrame(rows).to_csv(history_path, index=False)\n        print(f"{tag} {epoch:02d}/{args.epochs} "\n              f"train={row[\'train_error\']:.2f}% val={row[\'val_error\']:.2f}% "\n              f"test={row[\'test_error\']:.2f}% elapsed={row[\'seconds\']:.0f}s", flush=True)\n    torch.save({k: v.detach().cpu() for k, v in model.state_dict().items()}, checkpoint)\n    return pd.DataFrame(rows)\n\n\n@torch.no_grad()\ndef failure_plot(model, ds, class_names, device, path):\n    loader = DataLoader(ds, batch_size=512, shuffle=False)\n    mistakes = []\n    model.eval()\n    for x, y in loader:\n        probs = model(x.to(device)).softmax(1).cpu()\n        preds = probs.argmax(1)\n        for i in (preds != y).nonzero(as_tuple=True)[0].tolist():\n            mistakes.append((x[i, 0], int(y[i]), int(preds[i]), float(probs[i, preds[i]])))\n            if len(mistakes) == 12:\n                break\n        if len(mistakes) == 12:\n            break\n    fig, axes = plt.subplots(3, 4, figsize=(12, 8))\n    for ax in axes.flat:\n        ax.axis(\'off\')\n    for ax, (x, truth, pred, confidence) in zip(axes.flat, mistakes):\n        ax.imshow(x, cmap=\'gray\')\n        ax.set_title(f\'True: {class_names[truth]}\\nPred: {class_names[pred]} ({confidence:.2f})\', fontsize=8)\n    fig.suptitle(\'First misclassified official-test images; not cherry-picked\')\n    fig.tight_layout()\n    fig.savefig(path, dpi=180)\n    plt.close(fig)\n\n\ndef summarize(hist, args, out):\n    hist.to_csv(out / \'history.csv\', index=False)\n    final = hist[hist.epoch == args.epochs].copy()\n    final[\'test_accuracy\'] = 100 - final.test_error\n    final.to_csv(out / \'final_results.csv\', index=False)\n    cols = [\'train_error\', \'val_error\', \'test_error\', \'gap_pp\', \'test_accuracy\', \'seconds\']\n    grouped = final.groupby([\'dataset\', \'drop_rate\'])[cols]\n    avg = grouped.mean().add_suffix(\'_mean\')\n    sd = grouped.std(ddof=1).add_suffix(\'_sd\')\n    summary = pd.concat([avg, sd], axis=1).reset_index()\n    summary.to_csv(out / \'summary.csv\', index=False)\n    chosen = {}\n    lines = [\'# Actual measured results\', \'\',\n             \'Final fixed epoch; rates selected using mean validation error only.\',\n             \'SD is sample standard deviation across seeds; one seed has no SD.\', \'\',\n             \'| Dataset | Drop rate | Train error % | Val error % | Test error % | Test SD | Gap pp |\',\n             \'|---|---:|---:|---:|---:|---:|---:|\']\n    for r in summary.itertuples():\n        sd_text = f\'{r.test_error_sd:.3f}\' if pd.notna(r.test_error_sd) else \'N/A\'\n        lines.append(f\'| {r.dataset} | {r.drop_rate:.1f} | {r.train_error_mean:.3f} | \'\n                     f\'{r.val_error_mean:.3f} | {r.test_error_mean:.3f} | {sd_text} | {r.gap_pp_mean:.3f} |\')\n    for name in args.datasets:\n        sub = summary[summary.dataset == name].sort_values(\'drop_rate\')\n        best = sub.sort_values([\'val_error_mean\', \'drop_rate\']).iloc[0]\n        chosen[name] = float(best.drop_rate)\n        lines += [\'\', f\'{name}: validation-selected drop rate = {best.drop_rate:.1f}.\']\n        if 0.0 in sub.drop_rate.values:\n            baseline = sub[sub.drop_rate == 0].iloc[0]\n            delta = baseline.test_error_mean - best.test_error_mean\n            lines.append(f\'Test-error change vs no dropout = {delta:.3f} percentage points \'\n                         \'(positive means lower error with the selected model).\')\n        curves = hist[hist.dataset == name]\n        fig, axes = plt.subplots(1, 2, figsize=(12, 4))\n        reference_rate = 0.5 if 0.5 in sub.drop_rate.values else chosen[name]\n        for rate in sorted(set([0.0, reference_rate])):\n            d = curves[np.isclose(curves.drop_rate, rate)]\n            if d.empty:\n                continue\n            mean = d.groupby(\'epoch\').mean(numeric_only=True)\n            axes[0].plot(mean.index, mean.train_error, \'--\', label=f\'d={rate:g} train (eval)\')\n            axes[0].plot(mean.index, mean.test_error, label=f\'d={rate:g} test\')\n            axes[1].plot(mean.index, mean.val_loss, label=f\'d={rate:g} validation\')\n        axes[0].set_ylabel(\'Classification error (%)\')\n        axes[1].set_ylabel(\'Cross-entropy loss\')\n        for ax in axes:\n            ax.set_xlabel(\'Epoch\'); ax.legend(); ax.grid(alpha=0.25)\n        fig.suptitle(f\'{name}: reduced-scale overfitting comparison (means across seeds)\')\n        fig.tight_layout(); fig.savefig(out / f\'{name}_overfitting.png\', dpi=200); plt.close(fig)\n        for keep_axis in [False, True]:\n            plot_data = sub.copy()\n            plot_data[\'x\'] = 1 - plot_data.drop_rate if keep_axis else plot_data.drop_rate\n            plot_data = plot_data.sort_values(\'x\')\n            fig, ax = plt.subplots(figsize=(7, 4))\n            for metric, label in [(\'train_error\', \'Train (eval mode)\'), (\'test_error\', \'Test\')]:\n                errors = plot_data[f\'{metric}_sd\']\n                ax.errorbar(plot_data.x, plot_data[f\'{metric}_mean\'],\n                            yerr=None if errors.isna().all() else errors.fillna(0),\n                            marker=\'o\', capsize=3, label=label)\n            ax.set_xlabel(\'Keep probability q = 1 - d\' if keep_axis else \'Drop probability d\')\n            ax.set_ylabel(\'Classification error (%)\')\n            ax.set_title(f\'{name}: fixed-width dropout ablation\')\n            ax.legend(); ax.grid(alpha=0.25); fig.tight_layout()\n            suffix = \'keep_probability\' if keep_axis else \'drop_rates\'\n            fig.savefig(out / f\'{name}_{suffix}.png\', dpi=200); plt.close(fig)\n    (out / \'RESULTS.md\').write_text(\'\\n\'.join(lines) + \'\\n\')\n    (out / \'selected_rates.json\').write_text(json.dumps(chosen, indent=2))\n    print(\'\\n\'.join(lines))\n    return chosen\n\n\ndef main():\n    p = argparse.ArgumentParser()\n    p.add_argument(\'--datasets\', nargs=\'+\', choices=[\'mnist\', \'fashion\'], default=[\'mnist\'])\n    p.add_argument(\'--rates\', nargs=\'+\', type=float, default=[i / 10 for i in range(8)])\n    p.add_argument(\'--seeds\', nargs=\'+\', type=int, default=[42, 43, 44])\n    p.add_argument(\'--epochs\', type=int, default=20)\n    p.add_argument(\'--train-size\', type=int, default=10000)\n    p.add_argument(\'--val-size\', type=int, default=5000)\n    p.add_argument(\'--test-size\', type=int, default=10000)\n    p.add_argument(\'--split-seed\', type=int, default=2026)\n    p.add_argument(\'--width\', type=int, default=256)\n    p.add_argument(\'--batch-size\', type=int, default=128)\n    p.add_argument(\'--lr\', type=float, default=0.001)\n    p.add_argument(\'--threads\', type=int, default=4)\n    p.add_argument(\'--device\', choices=[\'auto\', \'cpu\', \'mps\', \'cuda\'], default=\'cpu\')\n    p.add_argument(\'--out\', default=\'results_mnist\')\n    p.add_argument(\'--self-test\', action=\'store_true\')\n    args = p.parse_args()\n    if any(v <= 0 for v in [args.epochs, args.train_size, args.val_size, args.test_size,\n                            args.width, args.batch_size, args.threads, args.lr]):\n        p.error(\'Sizes, epochs, threads and learning rate must be positive.\')\n    if any(not 0 <= r <= 0.7 or abs(r * 10 - round(r * 10)) > 1e-6 for r in args.rates):\n        p.error(\'Use rates 0.0, 0.1, ..., 0.7.\')\n    if len(set(args.seeds)) != len(args.seeds) or len(set(args.rates)) != len(args.rates):\n        p.error(\'No duplicate seeds or rates.\')\n    torch.set_num_threads(args.threads)\n    device = choose_device(args.device)\n    print(\'Device:\', device, \'| torch:\', torch.__version__)\n    self_test(device)\n    if args.self_test:\n        return\n    out = Path(args.out)\n    out.mkdir(parents=True, exist_ok=True)\n    config = vars(args).copy()\n    config.update(device_used=str(device), python=platform.python_version(),\n                  torch=str(torch.__version__), torchvision=str(torchvision.__version__),\n                  platform=platform.platform(), numpy=np.__version__, pandas=pd.__version__,\n                  matplotlib=matplotlib.__version__,\n                  source_sha256=hashlib.sha256(Path(__file__).read_bytes()).hexdigest())\n    config_path = out / \'config.json\'\n    if config_path.exists() and json.loads(config_path.read_text()) != config:\n        raise RuntimeError(\'Existing output has a different config/code/environment. Use a NEW --out folder.\')\n    config_path.write_text(json.dumps(config, indent=2))\n    print(\'Parameters:\', sum(v.numel() for v in MLP(0, args.width).parameters()))\n    histories, loaded = [], {}\n    for name in args.datasets:\n        sets, classes = load_data(name, args.train_size, args.val_size, args.test_size,\n                                  args.split_seed, out)\n        loaded[name] = (sets, classes)\n        for seed in args.seeds:\n            for rate in args.rates:\n                histories.append(train_one(name, rate, seed, sets, args, device, out))\n    selected = summarize(pd.concat(histories, ignore_index=True), args, out)\n    for name, rate in selected.items():\n        seed = args.seeds[0]\n        model = MLP(rate, args.width).to(device)\n        model.load_state_dict(torch.load(out / f\'{name}_d{rate:.1f}_s{seed}.pt\',\n                                         map_location=device, weights_only=True))\n        sets, classes = loaded[name]\n        failure_plot(model, sets[2], classes, device, out / f\'{name}_failure_cases.png\')\n    print(\'Finished. Results saved in\', out.resolve())\n\n\nif __name__ == \'__main__\':\n    main()\n')


In [ ]:
import subprocess, sys
# Run ManualDropout mathematical self-test on CPU
subprocess.run([sys.executable, "run.py", "--self-test", "--device", "cpu"], check=True)


## Main experiment — MNIST (Default Reproduction)

Executes the full 20-epoch × 8-rate × 3-seed sweep on real MNIST on CPU.
- Rates: `0.0, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7`
- Seeds: `42, 43, 44`
- Epochs: `20`
- Output directory: `results_mnist_colab`

In [ ]:
import subprocess, sys
from pathlib import Path

DATASETS = ["mnist"]
OUT = "results_mnist_colab"
DEVICE = "cpu"
command = [
    sys.executable, "run.py",
    "--datasets", *DATASETS,
    "--device", DEVICE,
    "--epochs", "20",
    "--seeds", "42", "43", "44",
    "--out", OUT
]
subprocess.run(command, check=True)


In [ ]:
import pandas as pd
from pathlib import Path
from IPython.display import display, Image, Markdown

out_path = Path(OUT)
if (out_path / "summary.csv").exists():
    display(Markdown("### MNIST Summary Table"))
    display(pd.read_csv(out_path / "summary.csv"))
    display(Markdown((out_path / "RESULTS.md").read_text()))
    for suffix in ["overfitting", "drop_rates", "keep_probability", "failure_cases"]:
        img_p = out_path / f"mnist_{suffix}.png"
        if img_p.exists():
            display(Image(filename=str(img_p)))


In [ ]:
import shutil
from pathlib import Path
if Path(OUT).exists():
    archive = shutil.make_archive(OUT, "zip", OUT)
    try:
        from google.colab import files
        files.download(archive)
    except ImportError:
        print("Saved local archive:", archive)


## Optional extension — Fashion-MNIST

Executes the identical frozen protocol (`dropout-24cs2032-v1`) on Fashion-MNIST as an extension benchmark.
To execute, uncomment the `subprocess.run(...)` call in the cell below.

In [ ]:
import subprocess, sys
from pathlib import Path

DATASETS_FASHION = ["fashion"]
OUT_FASHION = "results_fashion_colab"
command_fashion = [
    sys.executable, "run.py",
    "--datasets", *DATASETS_FASHION,
    "--device", DEVICE,
    "--epochs", "20",
    "--seeds", "42", "43", "44",
    "--out", OUT_FASHION
]
# Uncomment to execute Fashion-MNIST extension sweep:
# subprocess.run(command_fashion, check=True)


## Verified local reference results

*Note: The values below are documented reference results obtained from the verified final Mac CPU experiments (`protocol: dropout-24cs2032-v1`). They are provided as reference text for comparison and are NOT fabricated cell execution outputs.*

Model selection used mean validation error only across seeds at fixed epoch 20. Test error was not used to choose the dropout rate.

### MNIST (Main Reproduction)
- **No-dropout baseline ($d=0.0$):** Test error $4.320 \pm 0.295\%$, Gap $4.117$ pp
- **Validation-selected rate ($d=0.4$):** Validation error $3.747 \pm 0.130\%$
- **Selected model test error:** $3.843 \pm 0.282\%$ (Test accuracy $96.157\%$)
- **Selected model generalization gap:** $3.520 \pm 0.295$ pp
- **Test-error change vs baseline:** $+0.477$ percentage points lower test error ($11.03\%$ relative error reduction)
- **Paper reference rate ($d=0.5$):** Test error $3.883 \pm 0.087\%$
- **Descriptive test-error-best rate:** $d=0.3$ ($3.820 \pm 0.118\%$; not used for selection)
- **Negative finding:** High dropout rates noticeably degraded performance ($d=0.6$: $4.380\%$, $d=0.7$: $5.110\%$ vs $4.320\%$ baseline).

### Fashion-MNIST (Extension Benchmark)
- **No-dropout baseline ($d=0.0$):** Test error $14.663 \pm 0.175\%$, Gap $7.533$ pp
- **Validation-selected rate ($d=0.2$):** Validation error $13.087 \pm 0.408\%$
- **Selected model test error:** $14.347 \pm 0.253\%$ (Test accuracy $85.653\%$)
- **Selected model generalization gap:** $5.763 \pm 0.165$ pp
- **Test-error change vs baseline:** $+0.317$ percentage points lower test error ($2.16\%$ relative error reduction)
- **Paper reference rate ($d=0.5$):** Test error $15.133 \pm 0.325\%$ ($0.470$ pp *worse* than baseline)
- **Descriptive test-error-best rate:** $d=0.1$ ($14.220 \pm 0.105\%$; not used for selection)
- **Negative finding:** Reference $d=0.5$ underperformed no-dropout baseline; severe degradation occurred at higher rates ($d=0.7$: $17.453\%$).

### Comparison with Figure 9(a) of Srivastava et al. (2014)
Plotting test error against keep probability $q = 1 - d$ reproduces the qualitative U-shaped curve observed in Figure 9(a): performance improves with moderate retention rates and deteriorates as $q$ falls below $0.5$ (high drop rates).

## References & Assistance

1. Srivastava, N., Hinton, G., Krizhevsky, A., Sutskever, I., & Salakhutdinov, R. (2014). "Dropout: A Simple Way to Prevent Neural Networks from Overfitting." *Journal of Machine Learning Research*, 15(56), 1929-1958. https://jmlr.org/papers/v15/srivastava14a.html
2. Author dropout project page: https://www.cs.toronto.edu/~nitish/dropout/
3. Historical Deepnet repository (consulted for historical context; not a dependency): https://github.com/nitishsrivastava/deepnet
4. PyTorch Dropout reference (scaling formula comparison only): https://docs.pytorch.org/docs/stable/generated/torch.nn.Dropout.html
5. Torchvision MNIST/FashionMNIST documentation: https://docs.pytorch.org/vision/stable/generated/torchvision.datasets.MNIST.html

**Assistance and Provenance Statement:**
ChatGPT and Antigravity were used for implementation guidance, code review, local execution automation, verification, and documentation support. All reported experiment results come from the recorded real-data runs in this repository; no synthetic result data were used. External assistance is disclosed according to the course policy.